# Apples vs Tomatoes Binary Image Classification (ESE)

This notebook follows the exact university ESE requirements for a complete binary image classification pipeline.

## Step 1 — Imports

In [ ]:
import warnings
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import micropip
await micropip.install("imbalanced-learn")

from skimage.feature import local_binary_pattern
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

warnings.filterwarnings("ignore")

## Step 2 — Preprocessing Function

In [ ]:
def preprocess_image(img):
    """Resize, blur, and create HSV/gray representations."""
    img = cv2.resize(img, (128, 128))        # NOT 5x5 — that is wrong
    img = cv2.medianBlur(img, 5)             # blur BEFORE colour conversion
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    return img, hsv, gray

## Step 3 — Augmentation Function

In [ ]:
def augment_image(img):
    """Return five augmented versions of the input image."""
    return [
        img,
        cv2.flip(img, 1),  # horizontal flip
        np.clip(img + 30, 0, 255),  # brighter
        np.clip(img - 30, 0, 255),  # darker
        img + np.random.normal(0, 10, img.shape)  # gaussian noise
    ]

## Step 4 — Visualisation Functions

In [ ]:
def show_sample_images(base_path, samples=4):
    """Display sample images for each class in a 2 x samples grid."""
    class_names = ["apples", "tomatoes"]
    plt.figure(figsize=(4 * samples, 6))
    idx = 1
    for class_name in class_names:
        class_dir = os.path.join(base_path, class_name)
        if not os.path.isdir(class_dir):
            continue
        image_files = [f for f in os.listdir(class_dir) if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))]
        for img_name in image_files[:samples]:
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.subplot(2, samples, idx)
            plt.imshow(img)
            plt.title(f"{class_name}")
            plt.axis("off")
            idx += 1
    plt.tight_layout()


def show_preprocessing_demo(img_path):
    """Show Original, Grayscale+Equalized, HSV Hue, and Canny Edges."""
    img = cv2.imread(img_path)
    if img is None:
        print("Image not found for preprocessing demo.")
        return
    img, hsv, gray = preprocess_image(img)
    edges = cv2.Canny(gray, 100, 200)

    plt.figure(figsize=(14, 4))
    plt.subplot(1, 4, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(gray, cmap="gray")
    plt.title("Grayscale + Equalized")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(hsv[:, :, 0], cmap="hsv")
    plt.title("HSV Hue Channel")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(edges, cmap="gray")
    plt.title("Canny Edges")
    plt.axis("off")

    plt.tight_layout()


def show_augmentation_demo(img_path):
    """Show all five augmented versions of the image."""
    img = cv2.imread(img_path)
    if img is None:
        print("Image not found for augmentation demo.")
        return
    augmented = augment_image(img)
    titles = ["Original", "Horizontal Flip", "Brighter", "Darker", "Gaussian Noise"]

    plt.figure(figsize=(18, 4))
    for i, (aug_img, title) in enumerate(zip(augmented, titles), start=1):
        aug_img = np.clip(aug_img, 0, 255).astype(np.uint8)
        aug_img = cv2.cvtColor(aug_img, cv2.COLOR_BGR2RGB)
        plt.subplot(1, 5, i)
        plt.imshow(aug_img)
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()


def show_contours(img_path):
    """Draw green contours on the image."""
    img = cv2.imread(img_path)
    if img is None:
        print("Image not found for contour demo.")
        return
    img_resized, hsv, gray = preprocess_image(img)
    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contour_img = img_resized.copy()
    cv2.drawContours(contour_img, contours, -1, (0, 255, 0), 2)

    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(contour_img, cv2.COLOR_BGR2RGB))
    plt.title("Contours")
    plt.axis("off")
    plt.tight_layout()


def show_sift_keypoints_demo(img_path):
    """Show SIFT keypoints using subplot(3, 4, 12)."""
    img = cv2.imread(img_path)
    if img is None:
        print("Image not found for SIFT demo.")
        return
    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create()
    keypoints = sift.detect(gray, None)
    sift_img = cv2.drawKeypoints(
        img, keypoints, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )

    plt.figure(figsize=(12, 9))
    plt.subplot(3, 4, 12)
    plt.imshow(cv2.cvtColor(sift_img, cv2.COLOR_BGR2RGB))
    plt.title("SIFT Keypoints")
    plt.axis("off")
    plt.tight_layout()

In [ ]:
# Optional visual demos (runs only if dataset exists)
base_path = "cv_dataset/train"

sample_image_path = None
if os.path.isdir(base_path):
    for cls_name in ["apples", "tomatoes"]:
        class_dir = os.path.join(base_path, cls_name)
        if os.path.isdir(class_dir):
            files = [f for f in os.listdir(class_dir) if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))]
            if files:
                sample_image_path = os.path.join(class_dir, files[0])
                break

if sample_image_path:
    show_sample_images(base_path, samples=4)
    show_preprocessing_demo(sample_image_path)
    show_augmentation_demo(sample_image_path)
    show_contours(sample_image_path)
    show_sift_keypoints_demo(sample_image_path)

## Step 5 — Feature Extraction (45 Dimensions)

In [ ]:
def extract_features(img, hsv, gray):
    """Extract 45-dimensional colour, texture, and shape features."""
    features = []

    hue_hist = cv2.calcHist([hsv], [0], None, [32], [0, 180]).flatten()
    hue_hist = hue_hist / (np.sum(hue_hist) + 1e-6)
    features.extend(hue_hist)

    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 11), range=(0, 10))
    lbp_hist = lbp_hist.astype("float")
    lbp_hist = lbp_hist / (lbp_hist.sum() + 1e-6)
    features.extend(lbp_hist)

    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        c = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(c)
        perimeter = cv2.arcLength(c, True)
    else:
        area = 0.0
        perimeter = 0.0

    area = np.clip(area, 0, 1e6)
    perimeter = np.clip(perimeter, 0, 1e6)
    circularity = (4 * np.pi * area) / (perimeter ** 2 + 1e-6)
    features.extend([area, perimeter, circularity])

    return np.array(features, dtype=np.float32)

## Step 6 — Load Dataset with Corruption Handling

In [ ]:
def load_dataset(base_path):
    """Load dataset with augmentation and five-layer corruption checks."""
    X = []
    y = []
    class_map = {"apples": 0, "tomatoes": 1}

    for class_name, label in class_map.items():
        class_dir = os.path.join(base_path, class_name)
        if not os.path.isdir(class_dir):
            print(f"Missing folder: {class_dir}")
            continue
        image_files = [f for f in os.listdir(class_dir) if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))]
        num = 0

        for filename in image_files:
            file_path = os.path.join(class_dir, filename)
            try:
                img = cv2.imread(file_path)
            except Exception as e:
                print("Corrupt Image", e)
                continue

            if img is None:
                print(f"Corrupt Image (None): {filename}")
                continue
            if img.size == 0:
                print(f"Corrupt Image (empty): {filename}")
                continue
            if len(img.shape) < 3 or img.shape[2] != 3:
                print(f"Corrupt Image (bad shape): {filename}")
                continue

            for candidate in augment_image(img):
                candidate = np.clip(candidate, 0, 255).astype(np.uint8)
                img_proc, hsv, gray = preprocess_image(candidate)
                features = extract_features(img_proc, hsv, gray)
                X.append(features)
                y.append(label)

            num += 1
            print("Number of images Done", num, end="")

        print(f"
Label {class_name} Done")

    X = np.array(X)
    y = np.array(y)

    mask = np.isfinite(X).all(axis=1)
    X = X[mask]
    y = y[mask]

    return X, y

## Step 7 — SMOTE + Train/Test Split

In [ ]:
base_path = "cv_dataset/train"
X, y = load_dataset(base_path)

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

## Step 8 — KNN Learning Curves

In [ ]:
k_values = [3, 4, 5, 6, 7, 8, 9]
plt.figure(figsize=(16, 8))

for k in k_values:
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance", metric="minkowski"))
    ])
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_resampled, y_resampled,
        train_sizes=np.linspace(0.1, 1.0, 6), cv=5
    )
    val_mean = val_scores.mean(axis=1)
    plt.plot(train_sizes, val_mean, marker="o", label=f"k = {k}")

plt.xlabel("Training Set Size")
plt.ylabel("Validation Accuracy")
plt.title("KNN Learning Curves for Different k")
plt.legend()
plt.grid(True)
plt.tight_layout()

## Step 9 — Model Comparison (5-Fold CV)

In [ ]:
models = {
    "KNN (k=4)": Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=4, weights="distance", metric="minkowski"))
    ]),
    "SVM (RBF, C=10)": Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=10, probability=True))
    ]),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=1000))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB()
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_resampled, y_resampled, cv=cv, scoring="accuracy")
    results[name] = (scores.mean(), scores.std())

print(f"{'Model':<25} {'Mean Acc':<10} {'Std':<10}")
for name, (mean_acc, std_acc) in results.items():
    print(f"{name:<25} {mean_acc:<10.4f} {std_acc:<10.4f}")

means = [results[name][0] for name in results]
best_idx = int(np.argmax(means))
colors = ["#3498db"] * len(results)
colors[best_idx] = "#e74c3c"

plt.figure(figsize=(10, 6))
plt.barh(list(results.keys()), means, color=colors)
for i, value in enumerate(means):
    plt.text(value + 0.01, i, f"{value:.3f}", va="center")
plt.xlabel("Mean CV Accuracy")
plt.ylabel("Model")
plt.title("Model Comparison (5-Fold CV)")
plt.tight_layout()

## Step 10 — Train KNN (k=4)

In [ ]:
k = 4
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=k, weights="distance", metric="minkowski"))
])

knn_pipeline.fit(X_train, y_train)
y_pred = knn_pipeline.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Apples", "Tomatoes"]))

## Step 11 — Train Best Model from CV

In [ ]:
best_model_name = max(results, key=lambda k: results[k][0])
best_pipeline = models[best_model_name]

best_pipeline.fit(X_train, y_train)
y_pred_best = best_pipeline.predict(X_test)
y_pred_prob = best_pipeline.predict_proba(X_test)[:, 1]

print(f"Best CV Model: {best_model_name}")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")

## Step 12 — Evaluation Plots

In [ ]:
# Confusion Matrix Display
plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_best, display_labels=["Apples", "Tomatoes"], cmap="Blues"
)
plt.title("Confusion Matrix (Best Model)")
plt.tight_layout()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="crimson", label=f"AUC = {roc_auc:.3f}")
plt.fill_between(fpr, tpr, color="crimson", alpha=0.2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (Best Model)")
plt.legend(loc="lower right")
plt.tight_layout()

# Learning Curve for Best Model
train_sizes, train_scores, val_scores = learning_curve(
    best_pipeline, X_resampled, y_resampled,
    train_sizes=np.linspace(0.1, 1.0, 6), cv=5
)
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

plt.figure(figsize=(7, 5))
plt.plot(train_sizes, train_mean, color="royalblue", label="Train")
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, color="royalblue", alpha=0.2)
plt.plot(train_sizes, val_mean, color="darkorange", label="Validation")
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, color="darkorange", alpha=0.2)
plt.xlabel("Training Set Size")
plt.ylabel("Accuracy")
plt.title("Learning Curve (Best Model)")
plt.legend()
plt.tight_layout()

# PCA 2D Scatter
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_resampled)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_ * 100

plt.figure(figsize=(7, 5))
plt.scatter(X_pca[y_resampled == 0, 0], X_pca[y_resampled == 0, 1], c="red", label="Apples", alpha=0.7)
plt.scatter(X_pca[y_resampled == 1, 0], X_pca[y_resampled == 1, 1], c="green", label="Tomatoes", alpha=0.7)
plt.xlabel(f"PC1 ({explained[0]:.2f}%)")
plt.ylabel(f"PC2 ({explained[1]:.2f}%)")
plt.title("PCA 2D Scatter Plot")
plt.legend()
plt.tight_layout()

# Final Test Accuracy Table
print(f"{'Model':<25} {'Test Acc':<10} {'Note'}")

final_results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    final_results[name] = accuracy_score(y_test, preds)

best_test_name = max(final_results, key=final_results.get)
for name, acc in final_results.items():
    note = "<- BEST" if name == best_test_name else ""
    print(f"{name:<25} {acc:<10.4f} {note}")

## Step 13 — Summary Table

| Stage | Description | Output |
| --- | --- | --- |
| Imports | Required libraries and installs | Ready environment |
| Preprocessing | Resize + blur + HSV/Gray | Cleaned images |
| Augmentation | 5 variants per image | Expanded dataset |
| Visualisation | Samples, preprocessing, contours, SIFT | Visual checks |
| Features | 32 Hue + 10 LBP + 3 shape | 45-dim vectors |
| Loading | Corruption handling + augmentation | X, y arrays |
| SMOTE + Split | Balanced data + 80/20 split | Train/Test sets |
| KNN Curves | Learning curves for k values | Model insight |
| Model CV | 7 models with 5-fold CV | Best model |
| KNN Train | k=4 KNN baseline | Confusion matrix + report |
| Best Model | Train + predict + proba | y_pred_best, y_pred_prob |
| Evaluation | CM, ROC, LC, PCA, final table | Full analysis |